In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from PIL import Image
from fastai.vision.all import load_learner
from mtrain.smallnet.unet.predict.strided import single
from mtrain.utils import show, overlay_mask_on_img, draw_grid_cv2
from mtrain.disk import DiskBooleanMask, DiskImage

OV = overlay_mask_on_img

In [ ]:
from fastai.vision.all import SegmentationDataLoaders, Resize, unet_learner, resnet18
MODEL_PATH = "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-v3/log/export_iter_14.pkl"
SIZE = 100
AREA_THRES = 5

learner = load_learner(MODEL_PATH)
learner50 = load_learner(

    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/iter_4_engulf_t009_more-skew-resnet18-50x50-v2/model.pkl"
)

from fastai.vision.all import *
from mtrain.smallnet.unet.train import get_dls, get_learner

# dls = get_dls(BATCH_SIZE, LOG_BASE, TILE_SIZE, DATA_DIR / "images", DATA_DIR / "masks" )

import torch
from torch.utils.data import TensorDataset, DataLoader

# This is a typical setup for images
# x images have 3 channels, 128x128
# y has 1 channel because I needed to
# predict segmentation

dls = SegmentationDataLoaders.from_label_func(
    "./",
    bs=1,
    fnames=[
        Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/100/26210846388517270/image.jpg'),
    ],
    label_func=lambda o:  o.parent / "mask.png",
    codes=np.array(["background", "trash"]),
    item_tfms=Resize(200),
)
learner200 = unet_learner(dls, resnet18)
learner200 = learner200.load("/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-sz-200/log/resnet18_smallunet_200x200_iter_8")


# learner200 = load_learner(
#     "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-sz-200/log/resnet18_smallunet_200x200_iter_8.pth"
# )

In [ ]:
litters = [
    Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/100/26210846388517270/image.jpg'),
    Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/933090527327250/image.jpg'),
    Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/904411971827702/image.jpg'),
    Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/409495874297166/image.jpg'),
    Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/471611377923092/image.jpg'),
    Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/1132072153938604/image.jpg'),
    Path('/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/318666179638135/image.jpg'),
]

In [ ]:
idx = 0

In [ ]:
img = DiskImage.load(litters[idx])
mask = single.strided_predict_unet_only_mask(img, 100, learner, [50])
mask50 = single.strided_predict_unet_only_mask(img, 50, learner50, [25])

In [ ]:
strided_mask200_0 = single.strided_predict_unet_only_mask(img, 200, learner200, [])
strided_mask200_1 = single.strided_predict_unet_only_mask(img[100:,100:], 200, learner200, [])

In [ ]:
show([OV(img, mask), OV(img[100:,100:], strided_mask200_1)], (30,30), ncols=2)

In [ ]:
hehe = learner200.predict(img[100:,100:][200:400,800:])[0].numpy().astype(bool)
show([img[100:,100:][200:400, 800:], hehe])

In [ ]:
show([overlaid200, OV(img, strided_mask200)], (30,30), ncols=2)

In [ ]:
overlaid = OV(img, mask)
overlaid50 = OV(img, mask50)
overlaid200 = OV(img, mask200)
# overlaid200 = draw_grid_cv2(overlaid200, 200)
show([img, overlaid, overlaid200], (20,20), ncols=3, axis="on")

In [ ]:
crop = img[200:400, 800:1000]
# crop = img[450:550, 800:900]
# crop = img[400:600, 800:1000]
# crop = cv2.resize(crop, (100,100), interpolation=cv2.INTER_AREA)
# plt.imshow(crop)
crop_mask = learner200.predict(crop)[0].numpy()
show([crop, crop_mask, OV(crop, crop_mask), overlaid200], ncols=2,axis="off")

In [ ]:
overlaid = OV(img, mask)
base_overlaid = draw_grid_cv2(overlaid, 200)
strided_overlaid = draw_grid_cv2(overlaid[10:,50:], SIZE)
show([img, base_overlaid], (20,20))